<a href="https://colab.research.google.com/github/rose6492/fastbook/blob/master/bear_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Lignes Colab uniquement : à laisser commentées pour Binder
# ============================================================

# !pip install -Uqq fastai fastbook voila ipywidgets
# !jupyter serverextension enable --sys-prefix voila

# !rm -rf fastbook
# !git clone https://github.com/rose6492/fastbook.git
# %cd fastbook
# !ls -lh models


# ============================================================
# Imports nécessaires pour Binder / Voilà
# ============================================================

from fastai.vision.all import *
import ipywidgets as widgets
from ipywidgets import VBox
from IPython.display import display
from pathlib import Path
import io


# ============================================================
# Chargement du modèle
# ============================================================

model_path = Path("models") / "bear_classifier.pkl"

if not model_path.exists():
    raise FileNotFoundError(f"Modèle introuvable : {model_path.resolve()}")

learn_inf = load_learner(model_path, cpu=True)


# ============================================================
# Interface
# ============================================================

btn_upload = widgets.FileUpload()
btn_run = widgets.Button(description="Classify")

out_pl = widgets.Output()
lbl_pred = widgets.Label()


def get_uploaded_image(upload_widget):
    # Ancienne version ipywidgets
    if hasattr(upload_widget, "data") and len(upload_widget.data):
        return PILImage.create(upload_widget.data[-1])

    # Nouvelle version ipywidgets
    value = upload_widget.value

    if isinstance(value, tuple) and len(value) > 0:
        content = value[-1]["content"]
        return PILImage.create(io.BytesIO(content))

    if isinstance(value, dict) and len(value) > 0:
        content = next(iter(value.values()))["content"]
        return PILImage.create(io.BytesIO(content))

    raise Exception("Aucune image uploadée")


def on_click_classify(change):
    img = get_uploaded_image(btn_upload)

    out_pl.clear_output()
    with out_pl:
        display(img.to_thumb(128, 128))

    pred, pred_idx, probs = learn_inf.predict(img)
    lbl_pred.value = f"Prediction: {pred}; Probability: {probs[pred_idx]:.04f}"


btn_run.on_click(on_click_classify)


VBox([
    widgets.Label("Select your bear!"),
    btn_upload,
    btn_run,
    out_pl,
    lbl_pred
])